In [1]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

"""email_event_count
email_send_count
email_view_count
external_email_count
internal_email_count
total_sent_email_bytes
external_email_bytes
emails_with_attachments
attachment_count
unique_external_recipients
unique_external_domains"""


# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"
PROCESSED = PROJECT_ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

EMAIL_FILE = RAW / "email.csv"
EMAIL_DAILY = PROCESSED / "email_daily_r62.parquet"

email_path = str(EMAIL_FILE).replace("\\", "/")
email_daily_path = str(EMAIL_DAILY).replace("\\", "/")

assert EMAIL_FILE.exists(), f"Missing file: {EMAIL_FILE}"

print("Raw Email :", EMAIL_FILE)
print("Output    :", EMAIL_DAILY)

print("\n✅ SUCCESS — Email Daily setup is ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Raw Email : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\raw\r6.2\email.csv
Output    : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\email_daily_r62.parquet

✅ SUCCESS — Email Daily setup is ready.


In [3]:
# =========================================================
# BUILD EMAIL DAILY
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

    WITH email_base AS (

        SELECT
            id,
            user,

            CAST(
                strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                ) AS DATE
            ) AS day,

            activity,

            TRY_CAST(size AS BIGINT) AS email_size,

            COALESCE(attachments, '') AS attachments,

            REPLACE(
                CONCAT_WS(
                    ';',
                    NULLIF("to", ''),
                    NULLIF(cc, ''),
                    NULLIF(bcc, '')
                ),
                ',',
                ';'
            ) AS recipients

        FROM read_csv_auto(
            '{email_path}',
            all_varchar = TRUE
        )

    ),

    recipient_rows AS (

        SELECT
            id,
            user,
            day,

            LOWER(
                TRIM(recipient)
            ) AS recipient

        FROM email_base,

        UNNEST(
            STRING_SPLIT(
                recipients,
                ';'
            )
        ) AS t(recipient)

        WHERE activity = 'Send'
          AND TRIM(recipient) <> ''

    ),

    external_recipients AS (

        SELECT DISTINCT
            id,
            user,
            day,
            recipient,

            SPLIT_PART(
                recipient,
                '@',
                2
            ) AS external_domain

        FROM recipient_rows

        WHERE recipient
              NOT LIKE '%@dtaa.com'

    ),

    external_events AS (

        SELECT DISTINCT
            id,
            user,
            day

        FROM external_recipients

    ),

    external_daily AS (

        SELECT
            user,
            day,

            COUNT(
                DISTINCT recipient
            ) AS unique_external_recipients,

            COUNT(
                DISTINCT external_domain
            ) AS unique_external_domains

        FROM external_recipients

        GROUP BY
            user,
            day

    )

    SELECT
        b.user,
        b.day,

        COUNT(*) AS email_event_count,

        SUM(
            CASE
                WHEN b.activity = 'Send'
                THEN 1
                ELSE 0
            END
        ) AS email_send_count,

        SUM(
            CASE
                WHEN b.activity = 'View'
                THEN 1
                ELSE 0
            END
        ) AS email_view_count,

        SUM(
            CASE
                WHEN b.activity = 'Send'
                 AND e.id IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS external_email_count,

        SUM(
            CASE
                WHEN b.activity = 'Send'
                 AND e.id IS NULL
                THEN 1
                ELSE 0
            END
        ) AS internal_email_count,

        COALESCE(
            SUM(
                CASE
                    WHEN b.activity = 'Send'
                    THEN b.email_size
                    ELSE 0
                END
            ),
            0
        ) AS total_sent_email_bytes,

        COALESCE(
            SUM(
                CASE
                    WHEN b.activity = 'Send'
                     AND e.id IS NOT NULL
                    THEN b.email_size
                    ELSE 0
                END
            ),
            0
        ) AS external_email_bytes,

        SUM(
            CASE
                WHEN b.activity = 'Send'
                 AND TRIM(b.attachments) <> ''
                THEN 1
                ELSE 0
            END
        ) AS emails_with_attachments,

        SUM(
            CASE
                WHEN b.activity = 'Send'
                 AND TRIM(b.attachments) <> ''
                THEN
                    1
                    + LENGTH(b.attachments)
                    - LENGTH(
                        REPLACE(
                            b.attachments,
                            ';',
                            ''
                        )
                    )
                ELSE 0
            END
        ) AS attachment_count,

        COALESCE(
            MAX(
                d.unique_external_recipients
            ),
            0
        ) AS unique_external_recipients,

        COALESCE(
            MAX(
                d.unique_external_domains
            ),
            0
        ) AS unique_external_domains

    FROM email_base b

    LEFT JOIN external_events e
        ON b.id = e.id
       AND b.user = e.user
       AND b.day = e.day

    LEFT JOIN external_daily d
        ON b.user = d.user
       AND b.day = d.day

    GROUP BY
        b.user,
        b.day

    ORDER BY
        b.user,
        b.day

)

TO '{email_daily_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert EMAIL_DAILY.exists()

print("Created:")
print(EMAIL_DAILY)

print("\n✅ SUCCESS — email_daily_r62.parquet created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\email_daily_r62.parquet

✅ SUCCESS — email_daily_r62.parquet created.


In [5]:
# =========================================================
# INSPECT EMAIL DAILY
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{email_daily_path}'
)
""").df()

sample = con.execute(f"""
SELECT *
FROM read_parquet(
    '{email_daily_path}'
)
ORDER BY user, day
LIMIT 10
""").df()

con.close()

print("=" * 100)
print("EMAIL DAILY SCHEMA")
print("=" * 100)

print(
    schema[
        ["column_name", "column_type"]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

print(
    sample.to_string(index=False)
)

print("\n✅ SUCCESS — Email Daily inspected.")

EMAIL DAILY SCHEMA
               column_name column_type
                      user     VARCHAR
                       day        DATE
         email_event_count      BIGINT
          email_send_count      DOUBLE
          email_view_count      DOUBLE
      external_email_count      DOUBLE
      internal_email_count      DOUBLE
    total_sent_email_bytes      DOUBLE
      external_email_bytes      DOUBLE
   emails_with_attachments      DOUBLE
          attachment_count      DOUBLE
unique_external_recipients      BIGINT
   unique_external_domains      BIGINT

SAMPLE
   user        day  email_event_count  email_send_count  email_view_count  external_email_count  internal_email_count  total_sent_email_bytes  external_email_bytes  emails_with_attachments  attachment_count  unique_external_recipients  unique_external_domains
AAB0162 2010-01-04                  9               4.0               5.0                   1.0                   3.0               2471150.0               51708.0    

In [7]:
# =========================================================
# EMAIL DAILY VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    MIN(day) AS first_day,

    MAX(day) AS last_day,

    SUM(
        email_event_count
    ) AS total_events,

    SUM(
        email_send_count
    ) AS total_send,

    SUM(
        email_view_count
    ) AS total_view,

    SUM(
        external_email_count
    ) AS external_email_events,

    SUM(
        internal_email_count
    ) AS internal_email_events,

    SUM(
        attachment_count
    ) AS total_attachments

FROM read_parquet(
    '{email_daily_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*) AS duplicate_user_days

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet(
        '{email_daily_path}'
    )

    GROUP BY
        user,
        day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


null_keys = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{email_daily_path}'
)

WHERE user IS NULL
   OR day IS NULL
""").fetchone()[0]


negative_values = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{email_daily_path}'
)

WHERE email_send_count < 0
   OR email_view_count < 0
   OR external_email_count < 0
   OR internal_email_count < 0
   OR total_sent_email_bytes < 0
   OR external_email_bytes < 0
   OR attachment_count < 0
""").fetchone()[0]

con.close()


print("=" * 100)
print("EMAIL DAILY VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nDuplicate user-days :", duplicates)
print("Null user/day       :", null_keys)
print("Negative values     :", negative_values)


# =========================================================
# EXPECTED RAW EVENT TOTALS
# =========================================================

total_send = int(
    summary.loc[0, "total_send"]
)

total_view = int(
    summary.loc[0, "total_view"]
)

external_events = int(
    summary.loc[0, "external_email_events"]
)


print("\nChecks:")

print(
    "Send events match known raw count:",
    total_send == 3_832_466
)

print(
    "View events match known raw count:",
    total_view == 7_162_491
)

print(
    "True external events match audit:",
    external_events == 1_656_078
)


assert duplicates == 0
assert null_keys == 0
assert negative_values == 0

assert total_send == 3_832_466
assert total_view == 7_162_491
assert external_events == 1_656_078


print(
    "\n✅ SUCCESS — EMAIL DAILY PASSED ALL VALIDATION CHECKS."
)

EMAIL DAILY VALIDATION
   rows  users  first_day   last_day  total_events  total_send  total_view  external_email_events  internal_email_events  total_attachments
1381557   4000 2010-01-02 2011-05-31    10994957.0   3832466.0   7162491.0              1656078.0              2176388.0          1719195.0

Duplicate user-days : 0
Null user/day       : 0
Negative values     : 0

Checks:
Send events match known raw count: True
View events match known raw count: True
True external events match audit: True

✅ SUCCESS — EMAIL DAILY PASSED ALL VALIDATION CHECKS.


In [41]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

"""file_event_count
file_copy_count
file_open_count
file_write_count
file_delete_count

usb_copy_count
from_removable_count

unique_usb_files
unique_usb_extensions"""


RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"
PROCESSED = PROJECT_ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

FILE_FILE = RAW / "file.csv"
USB_DAILY = PROCESSED / "usb_daily_r62.parquet"

file_path = str(FILE_FILE).replace("\\", "/")
usb_daily_path = str(USB_DAILY).replace("\\", "/")

assert FILE_FILE.exists(), f"Missing file: {FILE_FILE}"

print("Raw File :", FILE_FILE)
print("Output   :", USB_DAILY)

print("\n✅ SUCCESS — USB/File Daily setup is ready.")

Raw File : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\raw\r6.2\file.csv
Output   : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\usb_daily_r62.parquet

✅ SUCCESS — USB/File Daily setup is ready.


In [12]:
# =========================================================
# BUILD USB / FILE DAILY
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

    WITH file_base AS (

        SELECT
            id,
            user,

            CAST(
                strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                ) AS DATE
            ) AS day,

            filename,
            activity,

            LOWER(
                COALESCE(
                    to_removable_media,
                    ''
                )
            ) AS to_removable,

            LOWER(
                COALESCE(
                    from_removable_media,
                    ''
                )
            ) AS from_removable,

            LOWER(
                REGEXP_EXTRACT(
                    filename,
                    '\\\\.([A-Za-z0-9]+)$',
                    1
                )
            ) AS file_extension

        FROM read_csv_auto(
            '{file_path}',
            all_varchar = TRUE
        )

    )

    SELECT
        user,
        day,

        COUNT(*) AS file_event_count,

        SUM(
            CASE
                WHEN activity = 'File Copy'
                THEN 1
                ELSE 0
            END
        ) AS file_copy_count,

        SUM(
            CASE
                WHEN activity = 'File Open'
                THEN 1
                ELSE 0
            END
        ) AS file_open_count,

        SUM(
            CASE
                WHEN activity = 'File Write'
                THEN 1
                ELSE 0
            END
        ) AS file_write_count,

        SUM(
            CASE
                WHEN activity = 'File Delete'
                THEN 1
                ELSE 0
            END
        ) AS file_delete_count,

        -- Files copied FROM computer TO removable media
        SUM(
            CASE
                WHEN activity = 'File Copy'
                 AND to_removable = 'true'
                THEN 1
                ELSE 0
            END
        ) AS usb_copy_count,

        -- File activity involving files coming FROM removable media
        SUM(
            CASE
                WHEN from_removable = 'true'
                THEN 1
                ELSE 0
            END
        ) AS from_removable_count,

        COUNT(
            DISTINCT CASE

                WHEN activity = 'File Copy'
                 AND to_removable = 'true'

                THEN filename

                ELSE NULL

            END
        ) AS unique_usb_files,

        COUNT(
            DISTINCT CASE

                WHEN activity = 'File Copy'
                 AND to_removable = 'true'
                 AND file_extension <> ''

                THEN file_extension

                ELSE NULL

            END
        ) AS unique_usb_extensions

    FROM file_base

    GROUP BY
        user,
        day

    ORDER BY
        user,
        day

)

TO '{usb_daily_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert USB_DAILY.exists()

print("Created:")
print(USB_DAILY)

print("\n✅ SUCCESS — usb_daily_r62.parquet created.")

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\usb_daily_r62.parquet

✅ SUCCESS — usb_daily_r62.parquet created.


In [14]:
# =========================================================
# INSPECT USB DAILY
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{usb_daily_path}'
)
""").df()

sample = con.execute(f"""
SELECT *
FROM read_parquet(
    '{usb_daily_path}'
)
ORDER BY user, day
LIMIT 10
""").df()

con.close()


print("=" * 100)
print("USB DAILY SCHEMA")
print("=" * 100)

print(
    schema[
        ["column_name", "column_type"]
    ].to_string(index=False)
)


print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

print(
    sample.to_string(index=False)
)

print("\n✅ SUCCESS — USB Daily inspected.")

USB DAILY SCHEMA
          column_name column_type
                 user     VARCHAR
                  day        DATE
     file_event_count      BIGINT
      file_copy_count      DOUBLE
      file_open_count      DOUBLE
     file_write_count      DOUBLE
    file_delete_count      DOUBLE
       usb_copy_count      DOUBLE
 from_removable_count      DOUBLE
     unique_usb_files      BIGINT
unique_usb_extensions      BIGINT

SAMPLE
   user        day  file_event_count  file_copy_count  file_open_count  file_write_count  file_delete_count  usb_copy_count  from_removable_count  unique_usb_files  unique_usb_extensions
AAB0162 2010-01-13                 2              0.0              2.0               0.0                0.0             0.0                   0.0                 0                      0
AAB0162 2010-01-18                 1              0.0              1.0               0.0                0.0             0.0                   0.0                 0                      0
AAB016

In [11]:
# =========================================================
# USB / FILE DAILY VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    MIN(day) AS first_day,

    MAX(day) AS last_day,

    SUM(
        file_event_count
    ) AS total_file_events,

    SUM(
        file_copy_count
    ) AS total_file_copy,

    SUM(
        file_open_count
    ) AS total_file_open,

    SUM(
        file_write_count
    ) AS total_file_write,

    SUM(
        file_delete_count
    ) AS total_file_delete,

    SUM(
        usb_copy_count
    ) AS total_usb_copy,

    SUM(
        from_removable_count
    ) AS total_from_removable,

    SUM(
        unique_usb_files
    ) AS summed_daily_unique_usb_files

FROM read_parquet(
    '{usb_daily_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet(
        '{usb_daily_path}'
    )

    GROUP BY
        user,
        day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


null_keys = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{usb_daily_path}'
)

WHERE user IS NULL
   OR day IS NULL
""").fetchone()[0]


negative_values = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{usb_daily_path}'
)

WHERE file_event_count < 0
   OR file_copy_count < 0
   OR file_open_count < 0
   OR file_write_count < 0
   OR file_delete_count < 0
   OR usb_copy_count < 0
   OR from_removable_count < 0
   OR unique_usb_files < 0
   OR unique_usb_extensions < 0
""").fetchone()[0]


impossible_usb = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{usb_daily_path}'
)

WHERE usb_copy_count > file_copy_count
""").fetchone()[0]


con.close()


print("=" * 100)
print("USB / FILE DAILY VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nDuplicate user-days :", duplicates)
print("Null user/day       :", null_keys)
print("Negative values     :", negative_values)
print("USB > all copies    :", impossible_usb)


# =========================================================
# KNOWN RAW COUNTS
# =========================================================

total_events = int(
    summary.loc[0, "total_file_events"]
)

total_copy = int(
    summary.loc[0, "total_file_copy"]
)

total_open = int(
    summary.loc[0, "total_file_open"]
)

total_write = int(
    summary.loc[0, "total_file_write"]
)

total_delete = int(
    summary.loc[0, "total_file_delete"]
)

total_usb = int(
    summary.loc[0, "total_usb_copy"]
)


print("\nChecks:")

print(
    "Total file events match:",
    total_events == 2_014_883
)

print(
    "File Copy match:",
    total_copy == 725_458
)

print(
    "File Open match:",
    total_open == 657_500
)

print(
    "File Write match:",
    total_write == 251_126
)

print(
    "File Delete match:",
    total_delete == 380_799
)

print(
    "USB Copy to removable match:",
    total_usb == 333_611
)


assert duplicates == 0
assert null_keys == 0
assert negative_values == 0
assert impossible_usb == 0

assert total_events == 2_014_883
assert total_copy == 725_458
assert total_open == 657_500
assert total_write == 251_126
assert total_delete == 380_799
assert total_usb == 333_611


print(
    "\n✅ SUCCESS — USB / FILE DAILY PASSED ALL VALIDATION CHECKS."
)

USB / FILE DAILY VALIDATION
  rows  users  first_day   last_day  total_file_events  total_file_copy  total_file_open  total_file_write  total_file_delete  total_usb_copy  total_from_removable  summed_daily_unique_usb_files
308647   3185 2010-01-02 2011-05-31          2014883.0         725458.0         657500.0          251126.0           380799.0        333611.0              971633.0                       285135.0

Duplicate user-days : 0
Null user/day       : 0
Negative values     : 0
USB > all copies    : 0

Checks:
Total file events match: True
File Copy match: True
File Open match: True
File Write match: True
File Delete match: True
USB Copy to removable match: True

✅ SUCCESS — USB / FILE DAILY PASSED ALL VALIDATION CHECKS.


In [25]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

"""http_event_count
www_visit_count
www_download_count
www_upload_count
unique_domains
unique_upload_domains """

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"
PROCESSED = PROJECT_ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

HTTP_FILE = RAW / "http.csv"
HTTP_DAILY = PROCESSED / "http_daily_r62.parquet"

http_path = str(HTTP_FILE).replace("\\", "/")
http_daily_path = str(HTTP_DAILY).replace("\\", "/")

assert HTTP_FILE.exists(), f"Missing file: {HTTP_FILE}"

print("Raw HTTP :", HTTP_FILE)
print("Output   :", HTTP_DAILY)

print("\n✅ SUCCESS — HTTP Daily setup is ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Raw HTTP : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\raw\r6.2\http.csv
Output   : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\http_daily_r62.parquet

✅ SUCCESS — HTTP Daily setup is ready.


In [27]:
# =========================================================
# BUILD HTTP DAILY
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

    WITH http_base AS (

        SELECT
            user,

            CAST(
                strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                ) AS DATE
            ) AS day,

            activity,

            NULLIF(
                LOWER(
                    REGEXP_EXTRACT(
                        url,
                        '^(?:https?://)?([^/]+)',
                        1
                    )
                ),
                ''
            ) AS domain

        FROM read_csv_auto(
            '{http_path}',
            all_varchar = TRUE
        )

    )

    SELECT
        user,
        day,

        COUNT(*) AS http_event_count,

        SUM(
            CASE
                WHEN activity = 'WWW Visit'
                THEN 1
                ELSE 0
            END
        ) AS www_visit_count,

        SUM(
            CASE
                WHEN activity = 'WWW Download'
                THEN 1
                ELSE 0
            END
        ) AS www_download_count,

        SUM(
            CASE
                WHEN activity = 'WWW Upload'
                THEN 1
                ELSE 0
            END
        ) AS www_upload_count,

        COUNT(
            DISTINCT domain
        ) AS unique_domains,

        COUNT(
            DISTINCT CASE
                WHEN activity = 'WWW Upload'
                THEN domain
                ELSE NULL
            END
        ) AS unique_upload_domains

    FROM http_base

    GROUP BY
        user,
        day

    ORDER BY
        user,
        day

)

TO '{http_daily_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert HTTP_DAILY.exists()

print("Created:")
print(HTTP_DAILY)

print("\n✅ SUCCESS — http_daily_r62.parquet created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\http_daily_r62.parquet

✅ SUCCESS — http_daily_r62.parquet created.


In [29]:
# =========================================================
# INSPECT HTTP DAILY
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{http_daily_path}'
)
""").df()

sample = con.execute(f"""
SELECT *
FROM read_parquet(
    '{http_daily_path}'
)
ORDER BY user, day
LIMIT 10
""").df()

con.close()

print("=" * 100)
print("HTTP DAILY SCHEMA")
print("=" * 100)

print(
    schema[
        ["column_name", "column_type"]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

print(
    sample.to_string(index=False)
)

print("\n✅ SUCCESS — HTTP Daily inspected.")

HTTP DAILY SCHEMA
          column_name column_type
                 user     VARCHAR
                  day        DATE
     http_event_count      BIGINT
      www_visit_count      DOUBLE
   www_download_count      DOUBLE
     www_upload_count      DOUBLE
       unique_domains      BIGINT
unique_upload_domains      BIGINT

SAMPLE
   user        day  http_event_count  www_visit_count  www_download_count  www_upload_count  unique_domains  unique_upload_domains
AAB0162 2010-01-04                95             95.0                 0.0               0.0              16                      0
AAB0162 2010-01-05                95             95.0                 0.0               0.0              14                      0
AAB0162 2010-01-06                95             95.0                 0.0               0.0              19                      0
AAB0162 2010-01-07                95             95.0                 0.0               0.0              26                      0
AAB0162 2010-

In [31]:
# =========================================================
# HTTP DAILY VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    MIN(day) AS first_day,

    MAX(day) AS last_day,

    SUM(
        http_event_count
    ) AS total_http_events,

    SUM(
        www_visit_count
    ) AS total_www_visit,

    SUM(
        www_download_count
    ) AS total_www_download,

    SUM(
        www_upload_count
    ) AS total_www_upload

FROM read_parquet(
    '{http_daily_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet(
        '{http_daily_path}'
    )

    GROUP BY
        user,
        day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


null_keys = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{http_daily_path}'
)

WHERE user IS NULL
   OR day IS NULL
""").fetchone()[0]


negative_values = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{http_daily_path}'
)

WHERE http_event_count < 0
   OR www_visit_count < 0
   OR www_download_count < 0
   OR www_upload_count < 0
   OR unique_domains < 0
   OR unique_upload_domains < 0
""").fetchone()[0]


impossible_activity = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{http_daily_path}'
)

WHERE
    www_visit_count
    + www_download_count
    + www_upload_count
    <> http_event_count
""").fetchone()[0]


upload_users = con.execute(f"""
SELECT
    COUNT(DISTINCT user)

FROM read_parquet(
    '{http_daily_path}'
)

WHERE www_upload_count > 0
""").fetchone()[0]


impossible_upload_domains = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{http_daily_path}'
)

WHERE unique_upload_domains > unique_domains
""").fetchone()[0]


con.close()


print("=" * 100)
print("HTTP DAILY VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nDuplicate user-days      :", duplicates)
print("Null user/day            :", null_keys)
print("Negative values          :", negative_values)
print("Activity total mismatch  :", impossible_activity)
print("Upload domains > domains :", impossible_upload_domains)
print("Users with WWW Upload    :", upload_users)


# =========================================================
# KNOWN RAW COUNTS
# =========================================================

total_events = int(
    summary.loc[0, "total_http_events"]
)

total_visit = int(
    summary.loc[0, "total_www_visit"]
)

total_download = int(
    summary.loc[0, "total_www_download"]
)

total_upload = int(
    summary.loc[0, "total_www_upload"]
)


print("\nChecks:")

print(
    "Total HTTP events match:",
    total_events == 117_025_216
)

print(
    "WWW Visit match:",
    total_visit == 115_401_178
)

print(
    "WWW Download match:",
    total_download == 1_413_637
)

print(
    "WWW Upload match:",
    total_upload == 210_401
)

print(
    "WWW Upload users match:",
    upload_users == 619
)


assert duplicates == 0
assert null_keys == 0
assert negative_values == 0
assert impossible_activity == 0
assert impossible_upload_domains == 0

assert total_events == 117_025_216
assert total_visit == 115_401_178
assert total_download == 1_413_637
assert total_upload == 210_401
assert upload_users == 619


print(
    "\n✅ SUCCESS — HTTP DAILY PASSED ALL VALIDATION CHECKS."
)

HTTP DAILY VALIDATION
   rows  users  first_day   last_day  total_http_events  total_www_visit  total_www_download  total_www_upload
1391239   4000 2010-01-02 2011-05-31        117025216.0      115401178.0           1413637.0          210401.0

Duplicate user-days      : 0
Null user/day            : 0
Negative values          : 0
Activity total mismatch  : 0
Upload domains > domains : 0
Users with WWW Upload    : 619

Checks:
Total HTTP events match: True
WWW Visit match: True
WWW Download match: True
WWW Upload match: True
WWW Upload users match: True

✅ SUCCESS — HTTP DAILY PASSED ALL VALIDATION CHECKS.


In [33]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"
PROCESSED = PROJECT_ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

LOGON_FILE = RAW / "logon.csv"
LOGON_DAILY = PROCESSED / "logon_daily_r62.parquet"

logon_path = str(LOGON_FILE).replace("\\", "/")
logon_daily_path = str(LOGON_DAILY).replace("\\", "/")

assert LOGON_FILE.exists(), f"Missing file: {LOGON_FILE}"

print("Raw Logon :", LOGON_FILE)
print("Output    :", LOGON_DAILY)

print("\n✅ SUCCESS — Logon Daily setup is ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Raw Logon : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\raw\r6.2\logon.csv
Output    : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\logon_daily_r62.parquet

✅ SUCCESS — Logon Daily setup is ready.


In [35]:
# =========================================================
# BUILD LOGON DAILY
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

    WITH logon_base AS (

        SELECT
            user,

            CAST(
                strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                ) AS TIMESTAMP
            ) AS event_time,

            CAST(
                strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                ) AS DATE
            ) AS day,

            activity

        FROM read_csv_auto(
            '{logon_path}',
            all_varchar = TRUE
        )

    )

    SELECT
        user,
        day,

        SUM(
            CASE
                WHEN activity = 'Logon'
                THEN 1
                ELSE 0
            END
        ) AS logon_count,

        SUM(
            CASE
                WHEN activity = 'Logoff'
                THEN 1
                ELSE 0
            END
        ) AS logoff_count,

        MIN(
            CASE
                WHEN activity = 'Logon'
                THEN event_time
                ELSE NULL
            END
        ) AS first_logon,

        MAX(
            CASE
                WHEN activity = 'Logoff'
                THEN event_time
                ELSE NULL
            END
        ) AS last_logoff

    FROM logon_base

    GROUP BY
        user,
        day

    ORDER BY
        user,
        day

)

TO '{logon_daily_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert LOGON_DAILY.exists()

print("Created:")
print(LOGON_DAILY)

print("\n✅ SUCCESS — logon_daily_r62.parquet created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\logon_daily_r62.parquet

✅ SUCCESS — logon_daily_r62.parquet created.


In [37]:
# =========================================================
# INSPECT LOGON DAILY
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{logon_daily_path}'
)
""").df()

sample = con.execute(f"""
SELECT *
FROM read_parquet(
    '{logon_daily_path}'
)
ORDER BY user, day
LIMIT 10
""").df()

con.close()

print("=" * 100)
print("LOGON DAILY SCHEMA")
print("=" * 100)

print(
    schema[
        ["column_name", "column_type"]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

print(
    sample.to_string(index=False)
)

print("\n✅ SUCCESS — Logon Daily inspected.")

LOGON DAILY SCHEMA
 column_name column_type
        user     VARCHAR
         day        DATE
 logon_count      DOUBLE
logoff_count      DOUBLE
 first_logon   TIMESTAMP
 last_logoff   TIMESTAMP

SAMPLE
   user        day  logon_count  logoff_count         first_logon         last_logoff
AAB0162 2010-01-04          1.0           1.0 2010-01-04 07:41:00 2010-01-04 18:46:00
AAB0162 2010-01-05          1.0           1.0 2010-01-05 07:46:00 2010-01-05 18:40:00
AAB0162 2010-01-06          1.0           1.0 2010-01-06 07:45:00 2010-01-06 18:55:00
AAB0162 2010-01-07          1.0           1.0 2010-01-07 07:45:00 2010-01-07 18:43:00
AAB0162 2010-01-08          1.0           1.0 2010-01-08 07:50:00 2010-01-08 18:41:00
AAB0162 2010-01-11          1.0           1.0 2010-01-11 07:33:00 2010-01-11 18:48:00
AAB0162 2010-01-12          1.0           1.0 2010-01-12 07:44:00 2010-01-12 18:58:00
AAB0162 2010-01-13          1.0           1.0 2010-01-13 07:50:00 2010-01-13 18:42:00
AAB0162 2010-01-14      

In [39]:
# =========================================================
# LOGON DAILY VALIDATION
# =========================================================

con = duckdb.connect()

raw_summary = con.execute(f"""
SELECT

    COUNT(*) AS total_events,

    SUM(
        CASE
            WHEN activity = 'Logon'
            THEN 1
            ELSE 0
        END
    ) AS total_logon,

    SUM(
        CASE
            WHEN activity = 'Logoff'
            THEN 1
            ELSE 0
        END
    ) AS total_logoff,

    COUNT(
        DISTINCT user
    ) AS users

FROM read_csv_auto(
    '{logon_path}',
    all_varchar = TRUE
)
""").df()


daily_summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    MIN(day) AS first_day,

    MAX(day) AS last_day,

    SUM(
        logon_count
    ) AS total_logon,

    SUM(
        logoff_count
    ) AS total_logoff

FROM read_parquet(
    '{logon_daily_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet(
        '{logon_daily_path}'
    )

    GROUP BY
        user,
        day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


null_keys = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{logon_daily_path}'
)

WHERE user IS NULL
   OR day IS NULL
""").fetchone()[0]


negative_values = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{logon_daily_path}'
)

WHERE logon_count < 0
   OR logoff_count < 0
""").fetchone()[0]

con.close()


print("=" * 100)
print("RAW LOGON SUMMARY")
print("=" * 100)

print(
    raw_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 100)
print("LOGON DAILY SUMMARY")
print("=" * 100)

print(
    daily_summary.to_string(
        index=False
    )
)


print("\nDuplicate user-days :", duplicates)
print("Null user/day       :", null_keys)
print("Negative values     :", negative_values)


raw_logon = int(
    raw_summary.loc[0, "total_logon"]
)

raw_logoff = int(
    raw_summary.loc[0, "total_logoff"]
)

daily_logon = int(
    daily_summary.loc[0, "total_logon"]
)

daily_logoff = int(
    daily_summary.loc[0, "total_logoff"]
)


print("\nChecks:")

print(
    "Logon count preserved:",
    raw_logon == daily_logon
)

print(
    "Logoff count preserved:",
    raw_logoff == daily_logoff
)


assert duplicates == 0
assert null_keys == 0
assert negative_values == 0

assert raw_logon == daily_logon
assert raw_logoff == daily_logoff


print(
    "\n✅ SUCCESS — LOGON DAILY PASSED ALL VALIDATION CHECKS."
)

RAW LOGON SUMMARY
 total_events  total_logon  total_logoff  users
      3530285    1948933.0     1581352.0   4000

LOGON DAILY SUMMARY
   rows  users  first_day   last_day  total_logon  total_logoff
1394010   4000 2010-01-02 2011-06-01    1948933.0     1581352.0

Duplicate user-days : 0
Null user/day       : 0
Negative values     : 0

Checks:
Logon count preserved: True
Logoff count preserved: True

✅ SUCCESS — LOGON DAILY PASSED ALL VALIDATION CHECKS.
